Load the train dataset and print the shape of an image and its label mask (ground truth)

In [1]:
from models.lcamazon import LCAmazon
import numpy as np

dataset=LCAmazon(root="DATA", modality="s2", split="train", aug_geometric=False)
print(f"Dataset of length {len(dataset)}")
img, label = dataset[0]
print(f"Image of shape: {np.shape(img)}, Label mask (ground truth) of shape: {np.shape(label)}")

Dataset of length 3840
Image of shape: (47, 47, 12), Label mask (ground truth) of shape: (47, 47)


# Visualisation + Check Data Augmentation
Let's now plot the satellite image together with groundtruth. We also plot the data augmented version below to check if both the groundtruth and the image are transformed correctly.

In [2]:
from ipywidgets import interact
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

aug=LCAmazon(root="DATA", modality="s2", split="train", aug_geometric=True)
@interact(idx=range(len(dataset)))
def plot_sample(idx=0):
    # Original sample
    img, label = dataset[idx]

    r,g,b   = img[:, :, 3], img[:, :, 2], img[:, :, 1]
    rgb = np.stack([r,g,b], axis=-1).astype(np.float32)
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # Augmented sample
    aug_img, aug_label = aug[idx]

    r_aug, g_aug, b_aug   = aug_img[:, :, 3], aug_img[:, :, 2], aug_img[:, :, 1]
    rgb_aug = np.stack([r_aug, g_aug, b_aug], axis=-1).astype(np.float32)
    rgb_aug /= np.percentile(rgb_aug, 99)
    rgb_aug = np.clip(rgb_aug, 0, 1)

    # Class mapping
    class_mapping = {
        new_id: class_name       # to deal with unused classes
        for class_name, old_id in LCAmazon.LABEL_CLASSES.items()
        if old_id in LCAmazon.LABEL_REMAP
        for new_id in [LCAmazon.LABEL_REMAP[old_id]]
    }

    unique_labels = np.unique(label)
    cmap = plt.cm.get_cmap("tab20", len(class_mapping))

    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(6, 6))

    # Original RGB
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Original RGB")
    axes[0, 0].axis("off")

    # Original GT
    axes[0, 1].imshow(label, cmap=cmap, vmin=0, vmax=len(class_mapping)-1)
    axes[0, 1].set_title("Original Ground Truth")
    axes[0, 1].axis("off")

    # Augmented RGB
    axes[1, 0].imshow(rgb_aug)
    axes[1, 0].set_title("Augmented RGB")
    axes[1, 0].axis("off")

    # Augmented GT
    axes[1, 1].imshow(aug_label, cmap=cmap, vmin=0, vmax=len(class_mapping)-1)
    axes[1, 1].set_title("Augmented Ground Truth")
    axes[1, 1].axis("off")

    # Legend
    legend_patches = [
        mpatches.Patch(color=cmap(class_id), label=class_mapping[class_id])
        for class_id in unique_labels
    ]

    axes[1, 1].legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0.
    )

    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

A function to plot prediction maps from all three models, ground truth and RGB satellite image all together
Qualitative assessmet

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from ipywidgets import interact

def plot(PRED_ROOT_S2, PRED_ROOT_AE, PRED_ROOT_AE_RF, dataset):
    REMAPPED_ID_TO_NAME = {LCAmazon.LABEL_REMAP[v]: k
        for k, v in LCAmazon.LABEL_CLASSES.items()}
    N_CLASSES = max(REMAPPED_ID_TO_NAME.keys()) 
    CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)

    def load_pred(root, fname):
        path = os.path.join(root, fname)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Prediction not found: {path}")
        with rasterio.open(path) as src:
            return src.read(1).astype(np.int32)

    @interact(idx=range(len(dataset)))
    def plot_all(idx=0):
        img, gt_label = dataset[idx]

        # RGB
        r,g,b   = img[:, :, 3], img[:, :, 2], img[:, :, 1]
        rgb = np.stack([r,g,b], axis=-1).astype(np.float32)
        rgb /= np.percentile(rgb, 99)
        rgb = np.clip(rgb, 0, 1)

        # File name
        _, gt_path = dataset.samples[idx]
        fname = os.path.basename(gt_path)

        # Prediction
        pred_s2 = load_pred(PRED_ROOT_S2, fname)
        pred_ae = load_pred(PRED_ROOT_AE, fname)
        pred_ae_rf = load_pred(PRED_ROOT_AE_RF, fname)

        # Plotting
        fig, axes = plt.subplots(1, 5, figsize=(20, 6))
        # RGB
        axes[0].imshow(rgb)
        axes[0].set_title("RGB")
        axes[0].axis("off")
        # Ground Truth
        axes[1].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")
        # S2 Prediction
        axes[2].imshow(pred_s2, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[2].set_title("Sentinel-2 Prediction")
        axes[2].axis("off")
        # AE Prediction
        axes[3].imshow(pred_ae, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[3].set_title("AE Prediction")
        axes[3].axis("off")
        # AE_RF Prediction
        axes[4].imshow(pred_ae_rf, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[4].set_title("AE_RF Prediction")
        axes[4].axis("off")
        # Legend
        unique_labels = np.unique(np.concatenate([np.unique(gt_label), np.unique(pred_s2), np.unique(pred_ae), np.unique(pred_ae_rf)]))
    
        legend_patches = []
        for class_id in unique_labels:
            if class_id == 0:
                name = "Background / Ignored"
            else:
                name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")
            legend_patches.append(mpatches.Patch(color=CMAP(class_id), label=name))

        fig.legend(handles=legend_patches,bbox_to_anchor=(1.05, 0.5),loc="center left")

        plt.tight_layout()
        plt.show()


All samples from training set

In [5]:
PRED_ROOT_S2 = "modeloutputs/s2_prediction"
PRED_ROOT_AE = "modeloutputs/AE_prediction"
PRED_ROOT_AE_RF = "modeloutputs/AE_RF_prediction"
train_all = LCAmazon(root="DATA", modality="s2", split="train_all", aug_geometric=False)
plot(PRED_ROOT_S2, PRED_ROOT_AE, PRED_ROOT_AE_RF, train_all)

/tmp/ipykernel_3999669/2626597877.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

All samples from test set

In [6]:
test = LCAmazon(root="DATA", modality="s2", split="test", aug_geometric=False)
plot(PRED_ROOT_S2, PRED_ROOT_AE, PRED_ROOT_AE_RF, test)

/tmp/ipykernel_3999669/2626597877.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…